# Final-state radiation

Charged particles in the final state can radiate photons. EvtGen simulates this final-state radiation (FSR) for decay channels that carry the `FSR` (or legacy `PHOTOS`) flag in the decay file. {py:class}`.Generator` uses the QED shower of Vincia, which is part of Pythia8, when it is created with `fsr=True`. PHOTOS++ is not included.

The default decay table flags $J/\psi \to \mu^+\mu^-$ for FSR:

In [ ]:
from pyevtgen import DECAY_FILE

lines = DECAY_FILE.read_text().splitlines()
start = lines.index("Decay J/psi")
print("\n".join(lines[start : start + 3]))

Because a process can hold only one {py:class}`.Generator`, comparing FSR on and off requires two processes. The function below generates decays in a fresh Python process and counts the photons that accompany the muon pair.

In [ ]:
import subprocess  # ruff: ignore[suspicious-subprocess-import]
import sys
from textwrap import dedent

import numpy as np


def count_fsr_photons(*, fsr: bool, n: int = 5000) -> np.ndarray:
    script = dedent(f"""
        from pyevtgen import Generator
        generator = Generator(fsr={fsr}, seed=0)
        for event in generator.generate("J/psi", n={n}):
            names = [d.name for d in event.daughters]
            if "mu+" in names and "mu-" in names:
                print(names.count("gamma"))
    """)
    result = subprocess.run(  # ruff: ignore[subprocess-without-shell-equals-true]
        [sys.executable, "-c", script], capture_output=True, check=True, text=True
    )
    lines = result.stdout.splitlines()
    return np.array([int(line) for line in lines if line.isdigit()])

In [ ]:
for fsr in [False, True]:
    photons = count_fsr_photons(fsr=fsr)
    fraction = np.mean(photons > 0)
    print(f"fsr={fsr}: {len(photons)} decays, {fraction:.1%} with photons")

With FSR, a sizable fraction of the muon pairs is accompanied by one or more photons, which are added to the daughters of the $J/\psi$.